In [68]:
# ========== Day2：可选——拉取另一个本地模型 ==========
# deepseek-r1:1.5b 体积小，适合本机试验；当前整行注释，默认不执行
# !ollama 拉 deepseek-r1:1.5b


In [69]:
# ========== 导入：网页抓取 + 本地模型调用要用的库 ==========

# 标准库 os：读环境变量（本格后面虽未直接用，常与 dotenv 搭配）
import os
# requests：用 HTTP 拉取网页、也常用于调 Ollama 原生 API
import requests
# load_dotenv：从 .env 加载密钥到环境变量（本笔记本后续主要走本地 Ollama）
from dotenv import load_dotenv
# BeautifulSoup：解析 HTML，抽出标题与正文文本
from bs4 import BeautifulSoup
# Markdown / display：在 Jupyter 里渲染模型返回的 Markdown 摘要
from IPython.display import Markdown, display
# OpenAI 客户端：本格导入备用；后面实际用的是 requests 直调 Ollama /api/chat
from openai import OpenAI


In [70]:
# ========== Website 类：抓取一个 URL，清洗出标题 + 正文 ==========

# 浏览器 User-Agent：有些站点会拦截默认 python-requests 头，伪装成 Chrome 更稳
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    def __init__(self,url):
        # 保存原始 URL，便于调试/追溯
        self.url = url
        # GET 下载页面 HTML；headers 带上上面的 User-Agent
        response = requests.get(url, headers=headers)
        # 用 html.parser 把字节/HTML 解析成可遍历的 DOM 树
        soup = BeautifulSoup(response.content,'html.parser')
        # 取 <title> 文本；没有标题时给占位字符串（保留英文，可能进 prompt）
        self.title = soup.title.string if soup.title else "No title found"
        # 删掉脚本/样式/图片/输入框等对「摘要正文」噪音大的节点
        for irrelevant in soup.body(["script","style","img","input"]):
            irrelevant.decompose()
        # 从 body 抽取纯文本：换行分隔、去掉多余空白
        self.text = soup.body.get_text(separator="\n",strip=True)


In [52]:
# ========== Ollama 原生聊天 API 常量 ==========

# Ollama 原生端点（不是 /v1）：POST JSON，字段见官方 /api/chat
OLLAMA_API = "http://localhost:11434/api/chat"
# 告诉服务端请求体是 JSON
HEADERS = {"Content-Type":"application/json"}
# 当前选用的本地模型名（需已 ollama pull）
MODEL = "llama3.2"
# 备选模型草稿：换成 deepseek 时取消注释并改 MODEL（保持原注释不动）
# 模型 =“deepseek-r1:1.5b”


In [53]:
# ========== 构造 user prompt：把网页标题+正文塞进摘要指令 ==========

def user_prompt(website):
    # 先写标题上下文，让模型知道「在看哪个站」
    prompt = f"You are looking at a website titled {website.title}"
    # 追加任务说明：要求用 markdown 做短摘要；若有新闻/公告也一并总结
    # （prompt 英文必须保留：这是发给模型的指令，改译可能改变行为）
    prompt += "\nThe contents of this website is as follows; \
    please provide a short summary of this website in markdown. \
    If it includes newsor announcements, then summarize these too.\n\n"
    # 再拼上抓取到的正文纯文本
    prompt += website.text
    return prompt


In [63]:
# ========== 实例化：抓取 CNN 国际版首页 ==========
# Website.__init__ 会立刻发 HTTP 请求并解析，可能较慢
we = Website("https://edition.cnn.com/")


In [64]:
# ========== 组装 messages：本练习只用 user（system 行被注释） ==========

message = [
    # 可选 system 角色草稿（保持注释，不改变当前行为）
    # {"role":"system","content":"你是智能助手"},
    # user：内容来自 user_prompt(we)，含摘要指令 + 网页正文
    {"role":"user","content":user_prompt(we)}
]


In [65]:
# ========== 组装 Ollama /api/chat 的 JSON payload ==========

payload = {
    # 模型名：与本机已安装模型一致
    "model": MODEL,
    # 对话消息列表
    "messages": message,
    # stream=False：等整段生成完再一次性返回 JSON（非流式）
    "stream":False
}


In [66]:
# ========== 调用本地 Ollama：POST /api/chat ==========
# json=payload 自动序列化；headers 声明 Content-Type
response = requests.post(OLLAMA_API,json=payload,headers=HEADERS)


In [67]:
# ========== 展示模型摘要 ==========
# 非流式响应结构大致：{"message": {"role":"assistant","content":"..."}, ...}
# 下面中文键名是错误草稿，保持注释不执行
# print(response.json()['消息']['内容'])
# 取出 assistant 的 content，用 Markdown 渲染
display(Markdown(response.json()['message']['content']))


<think>
Alright, let me break this down step by step. The user provided a detailed query about the Breaking News website titled "Breaking News, Latest News and Videos | CNN." They want a short summary in markdown, including relevant news or announcements.

First, I need to identify the key sections of the content on the website. The main areas are breaking news, events, culture, health, education, technology, climate, weather, etc. Each of these categories has several news points that contribute to the overall narrative.

Looking at the user's query, they specifically mentioned asking for the ad feedback section and including any technical issues. From the content provided, there are several ads listed on the homepage. The user might be interested in understanding how CNN handles its advertisements and what users might expect from them.

I'll need to extract the main points from each relevant section. For breaking news, I can mention notable events like the Ukraine-Russia war or Israel-Hamas war. In culture, I'll note that the website promotes diversity, especially with content about athletes in Mexico City.

For health, there are mentions of Tekken, a video game related to autism, and issues with vaccines, which are crucial for public health awareness. Education is covered by topics on the arts, tech, and science, which likely target various audiences.

Technology news includes updates on AI and blockchain, which are hot areas in today's industry. Climate and weather sections highlight natural disasters like hurricanes, a significant global issue, and geopolitical events.

Weather specifically mentions hurricane impacts and climate change impacts, which are essential for meteorological awareness. Environmental and safety content focuses on sustainability and disaster management.

The user also mentioned video topics, including rocket launches and tech reviews. These can be useful for viewers looking to stay updated in their areas of interest.

Putting it all together, the summary should reflect each section's key points concisely, ensuring that all major news items are included without getting bogged down by minor details. It should give a clear overview of the website's content and what users might want to read.
</think>

Breaking News, Latest News and Videos | CNN  

The Breaking News website "Breaking News, Latest News and Videos | CNN" offers a wide range of news updates, videos, and culture-focused content. Below is a concise summary of its key sections:  

1. **Breakings**  
   - The site highlights significant events, including Ukraine-Russia War updates, Israel-Hamas War developments, and notable athletes in Mexico City.  

2. **Cultures**  
   - Promotes diversity and inclusion, with content on diverse topics such as the beauty of diversity, tech innovations, and cultural heritage.  

3. **Health**  
   - Features news related to diseases like Tekken (a video game based on autism), vaccine hesitancies, and health issues affecting the population.  

4. **Education**  
   - Provides information on trending topics in education, including art, technology, science, and sustainability.  

5. **Technology**  
   - Discusses advancements in AI, blockchain, and other emerging technologies relevant to modern society.  

6. **Climate and Weather**  
   - Highlights natural disasters like hurricanes, climate change impacts, and weather-related news such as tornadoes.  

7. **Video**  
   - Offers a variety of video content, including rocket launches, tech reviews, and sports highlights from popular events.  

8. **Vibe**  
   - Focuses on environmental topics like renewable energy, sustainability, and disaster management efforts.  

The site is a comprehensive resource for staying informed about current events, technology news, culture, health, and more.

In [45]:
# ========== 再抓一次同一站点（练习/调试用第二实例） ==========
ed = Website("https://edition.cnn.com/")
